# Extra metrics — Latin-script controls

This notebook computes the Latin-script control analyses used in the paper extension on orthographic sensitivity in neural MT evaluation. It mirrors the structure and tone of the Indic notebook: each section begins with a short markdown explanation, followed by a self-contained code cell that computes the result, saves a CSV, and silently cross-checks the key summary values against the paper references.

The notebook covers four additional diagnostics for the ENG→DEU and ENG→SPA control settings: MATTR, Byte Premium, a tokenizer-based compound proxy, and the fraction of unique word types that receive single-token XLM-R encoding. It also assembles a final summary table matching the paper's Latin-control discussion.

## Imports and configuration

This cell loads the libraries used throughout the notebook and defines the output directory and helper utilities. The verification references are kept internal so the notebook remains reusable even if section numbering or draft placement changes later.

In [ ]:
import os
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')

OUT = Path('latin_outputs')
OUT.mkdir(exist_ok=True)

_MATTR_REF  = {'DEU': 0.634, 'SPA': 0.581}
_BP_REF     = {'DEU': 1.25,  'SPA': 1.09}
_STOK_REF   = {'DEU': 14.6,  'SPA': 22.1}
_IP_REF     = {'DEU': 0.535, 'SPA': 1.009}
_TP_REF     = {'DEU': 1.133, 'SPA': 1.090}
_SBI_REF    = {'DEU': 2.24,  'SPA': 1.12}
_ICOMET_REF = {'DEU': -0.262,'SPA': -0.021}

def _flag(computed, reference, tol=0.015):
    return '✓' if abs(computed - reference) / max(abs(reference), 1e-9) <= tol else '~'

def parse_token_list(token_str):
    if pd.isna(token_str) or str(token_str).strip() == '':
        return []
    return [t.strip() for t in str(token_str).split('|') if t.strip()]

def word_token_pairs(token_list):
    pairs = []
    current_tokens = []
    for tok in token_list:
        if tok.startswith('▁') and current_tokens:
            surface = ''.join(current_tokens).replace('▁', '').strip()
            if surface:
                pairs.append((surface, len(current_tokens)))
            current_tokens = [tok]
        elif not current_tokens:
            current_tokens = [tok]
        else:
            current_tokens.append(tok)
    if current_tokens:
        surface = ''.join(current_tokens).replace('▁', '').strip()
        if surface:
            pairs.append((surface, len(current_tokens)))
    return pairs

def compute_mattr(texts, window=500):
    words = []
    for text in texts:
        if pd.notna(text):
            words.extend(str(text).lower().split())
    n = len(words)
    if n == 0:
        return 0.0
    if n < window:
        return len(set(words)) / n
    scores = []
    for i in range(n - window + 1):
        window_words = words[i:i+window]
        scores.append(len(set(window_words)) / window)
    return float(np.mean(scores))


## Load Latin control data

This cell loads the cleaned ENG→DEU and ENG→SPA control datasets. These are the Latin-script comparison settings used to separate script-surface effects from tokenizer–morphology mismatch.

In [ ]:
print('=' * 65)
print('LOADING DATA')
print('=' * 65)

candidate_dirs = [
    Path('../../data/processed/latin'),
    Path('../data/processed/latin'),
    Path('.'),
]

de_path = None
es_path = None
for base in candidate_dirs:
    d = base / 'tp_ip_ende_clean.csv'
    s = base / 'tp_ip_enes_clean.csv'
    if d.exists() and s.exists():
        de_path, es_path = d, s
        break

if de_path is None:
    raise FileNotFoundError('Could not locate tp_ip_ende_clean.csv and tp_ip_enes_clean.csv')

de = pd.read_csv(de_path)
es = pd.read_csv(es_path, encoding='utf-8', on_bad_lines='skip')

datasets = {
    'DEU': {'label': 'EN-DE (German)',  'script': 'Latin', 'family': 'Germanic', 'df': de},
    'SPA': {'label': 'EN-ES (Spanish)', 'script': 'Latin', 'family': 'Romance',  'df': es},
}

for code, meta in datasets.items():
    print(f"{meta['label']}: {meta['df'].shape[0]:,} rows")


## MATTR

This cell computes moving-average type-token ratio using whitespace tokenisation with a 500-token window (Covington & McFall, 2010). The measure is a vocabulary-free proxy for surface-form diversity that sits outside the XLM-R tokenizer, making it suitable for cross-script comparison. Results should reproduce DEU 0.634 and SPA 0.581 from the corpus-level corroboration table.

In [ ]:
print('\n' + '=' * 65)
print('ANALYSIS 1: MATTR (window=500)')
print('=' * 65)

mattr_rows = []

for code, meta in datasets.items():
    df = meta['df']
    mattr = compute_mattr(df['target'].dropna(), window=500)
    words = []
    for text in df['target'].dropna():
        words.extend(str(text).lower().split())
    simple_ttr = len(set(words)) / len(words) if words else 0.0
    row = {
        'Language'      : meta['label'],
        'Code'          : code,
        'MATTR_w500'    : round(mattr, 4),
        'Simple_TTR'    : round(simple_ttr, 4),
        'N_tokens_total': len(words),
        'N_unique_types': len(set(words)),
        'Check'         : _flag(round(mattr, 3), _MATTR_REF[code]),
    }
    mattr_rows.append(row)
    print(f"\n  {meta['label']}")
    print(f"  MATTR (window=500): {mattr:.4f}  {row['Check']}")
    print(f"  Simple TTR:         {simple_ttr:.4f}")

mattr_df = pd.DataFrame(mattr_rows)
mattr_df.to_csv(OUT / 'latin_mattr.csv', index=False)
print('\n✓ Saved: latin_mattr.csv')


## Byte Premium

This cell computes the UTF-8 bytes-per-word ratio of target text relative to the English source (Arnett & Bergen, 2025). For Latin-script languages the premium is close to 1.0 because all characters occupy a single byte, so byte overhead cannot account for the tokenization gap between DEU (TP 1.133) and SPA (TP 1.090). The analysis confirms that the explanation must be morphological rather than orthographic.

In [ ]:
print('\n' + '=' * 65)
print('ANALYSIS 2: BYTE PREMIUM vs English source')
print('=' * 65)

byte_rows = []

for code, meta in datasets.items():
    df = meta['df']
    tgt_bpw, src_bpw = [], []
    for text in df['target'].dropna():
        for word in str(text).split():
            tgt_bpw.append(len(word.encode('utf-8')))
    for text in df['source'].dropna():
        for word in str(text).split():
            src_bpw.append(len(word.encode('utf-8')))
    tgt_mean = float(np.mean(tgt_bpw))
    src_mean = float(np.mean(src_bpw))
    premium  = tgt_mean / src_mean
    row = {
        'Language'                  : meta['label'],
        'Code'                      : code,
        'Mean_bytes_per_word_target': round(tgt_mean, 4),
        'Mean_bytes_per_word_source': round(src_mean, 4),
        'Byte_premium'              : round(premium, 4),
        'Check'                     : _flag(round(premium, 2), _BP_REF[code]),
    }
    byte_rows.append(row)
    print(f"\n  {meta['label']}")
    print(f"  Mean bytes/word target: {tgt_mean:.4f}")
    print(f"  Mean bytes/word source: {src_mean:.4f}")
    print(f"  Byte premium:           {premium:.4f}×  {row['Check']}")

byte_df = pd.DataFrame(byte_rows)
byte_df.to_csv(OUT / 'latin_byte_premium.csv', index=False)
print('\n✓ Saved: latin_byte_premium.csv')


## Single-token coverage

This cell measures what fraction of unique target word types are typically encoded as a single XLM-R token. The analysis operates at the type level rather than the token-occurrence level because the paper discussion frames vocabulary coverage as a structural property of the lexicon, not a frequency-weighted average. Expected values: DEU 14.6 %, SPA 22.1 %.

In [ ]:
print('\n' + '=' * 65)
print('ANALYSIS 3: SINGLE-TOKEN VOCABULARY COVERAGE')
print('=' * 65)

single_rows = []

for code, meta in datasets.items():
    df = meta['df']
    word_tok_dict = {}
    for token_str in df['target_xlmr_tokens'].dropna():
        for surface, n_tok in word_token_pairs(parse_token_list(token_str)):
            word = surface.lower().strip()
            if word and len(word) >= 2:
                word_tok_dict.setdefault(word, []).append(n_tok)
    single_tok_types = sum(
        1 for counts in word_tok_dict.values()
        if Counter(counts).most_common(1)[0][0] == 1
    )
    total_types = len(word_tok_dict)
    pct_single  = 100 * single_tok_types / total_types
    row = {
        'Language'          : meta['label'],
        'Code'              : code,
        'Total_unique_types': total_types,
        'Single_token_types': single_tok_types,
        'Pct_single_token'  : round(pct_single, 2),
        'Multi_token_types' : total_types - single_tok_types,
        'Pct_multi_token'   : round(100 - pct_single, 2),
        'Check'             : _flag(round(pct_single, 1), _STOK_REF[code]),
    }
    single_rows.append(row)
    print(f"\n  {meta['label']}")
    print(f"  Total unique word types: {total_types:,}")
    print(f"  Single-token types:      {single_tok_types:,} ({pct_single:.2f}%)  {row['Check']}")
    print(f"  Multi-token types:       {total_types - single_tok_types:,} ({100 - pct_single:.2f}%)")

single_df = pd.DataFrame(single_rows)
single_df.to_csv(OUT / 'latin_single_token_coverage.csv', index=False)
print('\n✓ Saved: latin_single_token_coverage.csv')


## Compound proxy

This cell builds a tokenizer-based proxy for morphological packing by measuring how often target words require three or more XLM-R tokens. For German the dominant source is productive compound-noun formation, which forces BPE to split otherwise plausible word forms; the cell also reports the full bucket distribution so the proxy can be inspected rather than treated as a black box.

In [ ]:
print('\n' + '=' * 65)
print('ANALYSIS 4: COMPOUND-WORD PROXY')
print('=' * 65)

compound_rows = []

for code, meta in datasets.items():
    df = meta['df']
    buckets = {1: 0, 2: 0, '3-5': 0, '6+': 0}
    total = 0
    for token_str in df['target_xlmr_tokens'].dropna():
        for surface, n_tok in word_token_pairs(parse_token_list(token_str)):
            if len(surface) >= 3:
                total += 1
                if   n_tok == 1: buckets[1] += 1
                elif n_tok == 2: buckets[2] += 1
                elif n_tok <= 5: buckets['3-5'] += 1
                else:            buckets['6+']  += 1
    pct_compound = 100 * (buckets['3-5'] + buckets['6+']) / total
    row = {
        'Language'          : meta['label'],
        'Code'              : code,
        'Total_words'       : total,
        'Pct_1_token'       : round(100 * buckets[1]       / total, 2),
        'Pct_2_tokens'      : round(100 * buckets[2]       / total, 2),
        'Pct_3to5_tokens'   : round(100 * buckets['3-5']   / total, 2),
        'Pct_6plus_tokens'  : round(100 * buckets['6+']    / total, 2),
        'Pct_compound_proxy': round(pct_compound, 2),
    }
    compound_rows.append(row)
    print(f"\n  {meta['label']}")
    print(f"  % words → 1 token:       {row['Pct_1_token']}%")
    print(f"  % words → 2 tokens:      {row['Pct_2_tokens']}%")
    print(f"  % words → 3-5 tokens:    {row['Pct_3to5_tokens']}%  ← compound proxy")
    print(f"  % words → 6+ tokens:     {row['Pct_6plus_tokens']}%  ← heavy fragmentation")
    print(f"  % compound proxy (3+):   {row['Pct_compound_proxy']}%")

compound_df = pd.DataFrame(compound_rows)
compound_df.to_csv(OUT / 'latin_compound_proxy.csv', index=False)
print('\n✓ Saved: latin_compound_proxy.csv')


## Word-length to token-count correlation

This section computes the Spearman correlation between surface word length in characters and the number of XLM-R tokens assigned to that word. A strong positive correlation in German but not Spanish would confirm that German's higher tokenization overhead is driven by long compound words rather than by uniform fragmentation.

In [ ]:
print('\n' + '=' * 65)
print('ANALYSIS 5: WORD-LENGTH → TOKEN-COUNT CORRELATION')
print('=' * 65)

corr_rows = []

for code, meta in datasets.items():
    df = meta['df']
    char_lens, tok_counts = [], []
    for token_str in df['target_xlmr_tokens'].dropna():
        for surface, n_tok in word_token_pairs(parse_token_list(token_str)):
            if 1 <= len(surface) <= 60:
                char_lens.append(len(surface))
                tok_counts.append(n_tok)
    rho, pval = spearmanr(char_lens, tok_counts)
    row = {
        'Language'                  : meta['label'],
        'Code'                      : code,
        'N_word_tokens'             : len(char_lens),
        'Spearman_rho'              : round(float(rho), 4),
        'p_value'                   : float(f'{pval:.2e}'),
        'Mean_tok_short_1to5'       : round(float(np.mean([n for c, n in zip(char_lens, tok_counts) if 1 <= c <= 5])), 3),
        'Mean_tok_medium_6to10'     : round(float(np.mean([n for c, n in zip(char_lens, tok_counts) if 6 <= c <= 10])), 3),
        'Mean_tok_long_11plus'      : round(float(np.mean([n for c, n in zip(char_lens, tok_counts) if c >= 11])), 3),
    }
    corr_rows.append(row)
    print(f"\n  {meta['label']}")
    print(f"  Spearman ρ (char_length vs n_tokens): {rho:.4f}  p={pval:.2e}")
    print(f"  Mean tokens — short words (1-5 chars):  {row['Mean_tok_short_1to5']}")
    print(f"  Mean tokens — medium words (6-10 chars): {row['Mean_tok_medium_6to10']}")
    print(f"  Mean tokens — long words (11+ chars):   {row['Mean_tok_long_11plus']}")

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(OUT / 'latin_word_token_correlation.csv', index=False)
print('\n✓ Saved: latin_word_token_correlation.csv')


## Summary table

This cell assembles the final Latin-control summary combining TP/IP statistics already in the dataset with the newly computed diagnostics. The verification flags are kept internal and printed only as ✓/~ so the notebook can be re-run against any draft version without manual cross-referencing.

In [ ]:
print('\n' + '=' * 65)
print('MASTER SUMMARY TABLE')
print('=' * 65)

mattr_map    = {r['Code']: r for r in mattr_rows}
byte_map     = {r['Code']: r for r in byte_rows}
single_map   = {r['Code']: r for r in single_rows}
compound_map = {r['Code']: r for r in compound_rows}

summary_rows = []
for code, meta in datasets.items():
    df = meta['df']
    ip_col  = next((c for c in ('target_xlmr_IP', 'IP') if c in df.columns), None)
    tp_col  = next((c for c in ('target_xlmr_TP', 'TP') if c in df.columns), None)
    sbi_col = next((c for c in ('SBI',) if c in df.columns), None)
    mean_ip  = float(df[ip_col].mean())  if ip_col  else float('nan')
    mean_tp  = float(df[tp_col].mean())  if tp_col  else float('nan')
    mean_sbi = float(df[sbi_col].mean()) if sbi_col else float('nan')
    row = {
        'Language'          : meta['label'],
        'Code'              : code,
        'Mean_IP'           : round(mean_ip,  4),
        'Mean_TP'           : round(mean_tp,  4),
        'Mean_SBI'          : round(mean_sbi, 4),
        'MATTR'             : mattr_map.get(code, {}).get('MATTR_w500', float('nan')),
        'Byte_premium'      : byte_map.get(code, {}).get('Byte_premium', float('nan')),
        'Pct_single_token'  : single_map.get(code, {}).get('Pct_single_token', float('nan')),
        'Pct_compound_proxy': compound_map.get(code, {}).get('Pct_compound_proxy', float('nan')),
        'Chk_IP'            : _flag(mean_ip,  _IP_REF[code]),
        'Chk_TP'            : _flag(mean_tp,  _TP_REF[code]),
        'Chk_SBI'           : _flag(mean_sbi, _SBI_REF[code]),
    }
    summary_rows.append(row)
    print(f"  {meta['label']:20s}  IP={mean_ip:.4f} {row['Chk_IP']}  TP={mean_tp:.4f} {row['Chk_TP']}  SBI={mean_sbi:.4f} {row['Chk_SBI']}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT / 'latin_summary.csv', index=False)
print('\n✓ Saved: latin_summary.csv')
